# Chapter 1 — Spherical mass and the effective potential

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/01_spherical_mass.ipynb)

**Goal.** Before integrating a single orbit, learn to *predict* its key properties:
the gravitational field of a spherical body, its circular speed, and the radii between
which a star oscillates.

**You will learn**
1. the shell theorem — why only the mass *inside* radius $r$ pulls on a star at $r$;
2. how density $\rho(r)$ determines enclosed mass $M(r)$, potential $\Phi(r)$ and circular speed $v_c(r)$;
3. the effective potential $\Phi_{\rm eff}(r)$ and how it fixes pericentre and apocentre;
4. the virial theorem as a global check on a self-gravitating system;
5. how to get all of the above from AGAMA, and check it against our own calculation.

**Units.** Throughout this chapter $G = 1$, and masses and lengths are in units of the
model's total mass $M$ and scale radius $b$. (Chapter 3 shows how to convert to kpc, km/s, M$_\odot$.)

**Prerequisites.** Newton's law of gravity, vectors, basic integrals; Python with `numpy` and `matplotlib`.
Sections 1.1–1.4 use only `numpy` and `scipy`: we build everything by hand first, so that we know
what AGAMA does for us. Section 1.5 then repeats the key results with AGAMA.

In [ ]:
import numpy as np                      # arrays and maths on whole arrays at once
import matplotlib.pyplot as plt         # plotting
from scipy.integrate import quad        # numerical integration of a function of one variable
from scipy.optimize import brentq, minimize_scalar   # root finding; minimisation in one variable

G = 1.0                                 # gravitational constant in our units

## 1.1 The shell theorem, checked numerically

Newton showed that a thin spherical shell of mass $m$ and radius $R$
- exerts **no net force** on a point inside it ($r < R$), and
- attracts a point outside it ($r > R$) **as if all its mass were at the centre**: $g = G m / r^2$.

Rather than take this on trust, we build a shell out of $N$ point masses and add up
their pulls directly. To spread points evenly over a sphere we use the
*Fibonacci lattice*: point $i$ sits at height $z_i = 1 - (2i+1)/N$ and azimuth
$\phi_i = i \times 137.5^\circ$ (the "golden angle"). Equal steps in $z$ give equal areas on the
sphere (Archimedes' hat-box theorem), and the golden angle stops points from lining up in columns.

The acceleration at point $\mathbf x$ due to masses $m_j$ at $\mathbf x_j$ is Newton's law summed
over all of them:

$$ \mathbf g(\mathbf x) = G \sum_j m_j \frac{\mathbf x_j - \mathbf x}{|\mathbf x_j - \mathbf x|^3} . $$

In [ ]:
def fibonacci_sphere(n, radius=1.0):
    """n points spread almost uniformly over a sphere; returns array of shape (n, 3)."""
    i = np.arange(n)                             # 0, 1, ..., n-1 — numpy works on all points at once
    z = 1 - (2 * i + 1) / n                      # heights equally spaced in (-1, 1)
    phi = i * np.pi * (3 - np.sqrt(5))           # golden angle (137.5 deg) in radians, times i
    s = np.sqrt(1 - z**2)                        # radius of the horizontal circle at height z
    # np.column_stack turns three length-n arrays into one (n, 3) array of (x, y, z) rows
    return radius * np.column_stack([s * np.cos(phi), s * np.sin(phi), z])


def acceleration(point, sources, masses):
    """Newtonian acceleration at `point` (3 numbers) due to point masses at `sources` (n x 3)."""
    d = sources - point                          # (n, 3): vectors from the point to each mass (broadcasting)
    r3 = np.sum(d**2, axis=1)**1.5               # (n,): |d|^3; axis=1 sums over x, y, z of each row
    # masses[:, None] turns shape (n,) into (n, 1) so it multiplies each row of d;
    # summing over axis=0 adds up the contributions of all n masses
    return G * np.sum(masses[:, None] * d / r3[:, None], axis=0)


N, m_shell, R = 20000, 1.0, 1.0
shell = fibonacci_sphere(N, R)
masses = np.full(N, m_shell / N)                 # N equal masses adding up to m_shell

# Test points along one line through the centre. We use a "random-looking" direction rather than
# an axis, so that test points do not sit exactly in line with lattice points.
direction = np.array([0.3, 0.5, 0.81]) / np.linalg.norm([0.3, 0.5, 0.81])   # unit vector
r_test = np.concatenate([np.linspace(0.05, 0.9, 12), np.linspace(1.1, 4, 12)])  # 12 inside, 12 outside
# radial component g_r = g . (unit vector); '@' is the dot product. Negative g_r means inward.
g_r = np.array([acceleration(r * direction, shell, masses) @ direction for r in r_test])

plt.figure(figsize=(6, 4))
plt.plot(r_test, -g_r, 'o', label=f'sum over {N} point masses')
rr = np.linspace(1.0, 4, 100)
plt.plot(rr, G * m_shell / rr**2, 'k-', label=r'$Gm/r^2$ (shell theorem)')
plt.plot([0, 1], [0, 0], 'k-')                   # shell theorem inside: zero force
plt.axvline(R, color='gray', ls=':')
plt.xlabel('distance from centre $r$'); plt.ylabel('inward pull $-g_r$')
plt.legend(); plt.title('Field of a thin shell of radius 1'); plt.show()

inside, outside = r_test < R, r_test > R         # boolean "masks" that select array elements
print('max |g| inside             :', np.max(np.abs(g_r[inside])))
print('max relative error outside :', np.max(np.abs(-g_r[outside] * r_test[outside]**2 / (G * m_shell) - 1)))
assert np.max(np.abs(g_r[inside])) < 1e-5
assert np.max(np.abs(-g_r[outside] * r_test[outside]**2 / (G * m_shell) - 1)) < 1e-5

**Figure 1.1.** Inward pull $-g_r$ against distance $r$ from the centre of a thin shell of radius 1, summed over 20 000 point masses (points), and the shell-theorem prediction: zero inside, $Gm/r^2$ outside (black lines). The dotted line marks the shell.

Inside the shell (Figure 1.1) the pulls cancel to about one part in a million; outside they add up to $Gm/r^2$
to the same accuracy. The residuals come from representing a smooth shell by a finite number of points.

**Consequence.** A spherical galaxy is a set of nested shells. A star at radius $r$ feels only
the shells inside it:

$$ g(r) = -\frac{G M(r)}{r^2}, \qquad M(r) = 4\pi \int_0^r \rho(r')\, r'^2\, dr'. $$

The potential $\Phi$ (with $\mathbf g = -\nabla\Phi$ and $\Phi \to 0$ at infinity) collects
both the inner mass and the outer shells, each of which contributes a constant $-Gm/R$ inside it:

$$ \Phi(r) = -\frac{G M(r)}{r} - 4\pi G \int_r^\infty \rho(r')\, r'\, dr'. $$

## 1.2 From density to potential: the Plummer sphere

The **Plummer sphere** (Plummer 1911) is the simplest smooth model of a star cluster or
galaxy core:

$$ \rho(r) = \frac{3M}{4\pi b^3}\left(1+\frac{r^2}{b^2}\right)^{-5/2}, \qquad
   M(r) = \frac{M r^3}{(r^2+b^2)^{3/2}}, \qquad
   \Phi(r) = -\frac{GM}{\sqrt{r^2+b^2}}. $$

We compute $M(r)$ and $\Phi(r)$ *numerically* from $\rho$ with the two integrals above and
compare them with these formulae. This is the pattern we will use throughout:
**a numerical method is trusted only after it reproduces a known answer.**

`scipy.integrate.quad(f, a, b)` integrates a Python function `f` from `a` to `b` adaptively
(it refines where `f` changes quickly) and returns `(value, error_estimate)`; `b` may be `np.inf`.
`lambda x: ...` defines a small unnamed function inline.

*In AGAMA:* `pot.enclosedMass(r)` and `pot.potential(points)` return $M(r)$ and $\Phi$ for any
model, and `agama.Potential(type='Multipole', density=my_density_function, symmetry='spherical')`
solves the Poisson equation for an arbitrary density profile (Chapter 3 onwards).

In [ ]:
M, b = 1.0, 1.0

def rho_plummer(r):
    """Plummer density; works for a single r or a numpy array of radii."""
    return 3 * M / (4 * np.pi * b**3) * (1 + r**2 / b**2)**-2.5

def mass_numeric(r, rho):
    """M(r) = 4 pi int_0^r rho(r') r'^2 dr'.  quad returns (value, error); [0] keeps the value."""
    return quad(lambda x: 4 * np.pi * rho(x) * x**2, 0, r)[0]

def phi_numeric(r, rho):
    """Phi(r) = -G M(r)/r - 4 pi G int_r^inf rho(r') r' dr'  (inner mass + outer shells)."""
    outer = quad(lambda x: rho(x) * x, r, np.inf)[0]
    return -G * mass_numeric(r, rho) / r - 4 * np.pi * G * outer

r = np.logspace(-2, 2, 60)        # 60 radii from 0.01 to 100, evenly spaced in log r
# quad handles one number at a time, so we loop over radii with a list comprehension
M_num = np.array([mass_numeric(x, rho_plummer) for x in r])
Phi_num = np.array([phi_numeric(x, rho_plummer) for x in r])
M_exact = M * r**3 / (r**2 + b**2)**1.5
Phi_exact = -G * M / np.sqrt(r**2 + b**2)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))   # one row of three panels; ax[0], ax[1], ax[2]
ax[0].loglog(r, rho_plummer(r), 'k-'); ax[0].set_ylabel(r'$\rho(r)$')
ax[1].loglog(r, M_exact, 'k-', lw=3, alpha=0.3, label='formula')
ax[1].loglog(r, M_num, 'r--', label='numerical'); ax[1].set_ylabel('$M(r)$'); ax[1].legend()
ax[2].semilogx(r, Phi_exact, 'k-', lw=3, alpha=0.3, label='formula')
ax[2].semilogx(r, Phi_num, 'r--', label='numerical'); ax[2].set_ylabel(r'$\Phi(r)$'); ax[2].legend()
for a in ax: a.set_xlabel('$r/b$')
plt.tight_layout(); plt.show()

err_M = np.max(np.abs(M_num / M_exact - 1))
err_Phi = np.max(np.abs(Phi_num / Phi_exact - 1))
print(f'max relative error: M(r) {err_M:.1e},  Phi(r) {err_Phi:.1e}')
assert err_M < 1e-6 and err_Phi < 1e-6

**Figure 1.2.** The Plummer sphere in model units: density $\rho(r)$ (left), enclosed mass $M(r)$ (middle) and potential $\Phi(r)$ (right), against $r/b$. Thick grey: formulae; red dashed: numerical integrals.

Note the shapes in Figure 1.2: the density is flat inside $b$ (a *core*) and falls as $r^{-5}$ outside;
the enclosed mass grows as $r^3$ in the core and saturates at $M$; the potential is finite
at the centre, $\Phi(0) = -GM/b$.

### Circular speed

A star on a circular orbit needs centripetal acceleration $v^2/r$ equal to the inward pull:

$$ v_c^2(r) = r\,\frac{d\Phi}{dr} = \frac{G M(r)}{r}. $$

(The second form holds only for spherical systems; for a disk, use the first. AGAMA has no
separate circular-speed function — Chapter 0 shows how to build it from `pot.force`.)

Compare three models of total mass 1:
- **point mass** (Kepler), $\Phi = -GM/r$: $v_c \propto r^{-1/2}$ at all radii;
- **homogeneous sphere** of radius $a$: $M(r) = M r^3/a^3$ inside, so $v_c \propto r$ — *solid-body* rotation;
- **Plummer**: solid-body in the core, Keplerian far outside, with a peak in between.

In [ ]:
def vc_kepler(r):      return np.sqrt(G * M / r)
# np.minimum(r, a): inside the sphere M(r) = M r^3/a^3; outside, all the mass M is enclosed
def vc_homog(r, a=1):  return np.sqrt(G * M * np.minimum(r, a)**3 / a**3 / r)
def vc_plummer(r):     return np.sqrt(G * M * r**2 / (r**2 + b**2)**1.5)

r = np.linspace(0.01, 6, 400)
plt.figure(figsize=(6, 4))
plt.plot(r, vc_kepler(r), label='point mass')
plt.plot(r, vc_homog(r), label='homogeneous sphere, $a=1$')
plt.plot(r, vc_plummer(r), label='Plummer, $b=1$')
plt.ylim(0, 1.5); plt.xlabel('$r$'); plt.ylabel('$v_c$'); plt.legend(); plt.show()

# Where does the Plummer curve peak?  Analytically: d(v_c^2)/dr = 0  =>  r = sqrt(2) b.
# minimize_scalar finds the minimum of a function, so we minimise -v_c within the bounds (0.1, 5);
# xatol sets how precisely the location is found; .x is the location of the minimum.
r_peak = minimize_scalar(lambda x: -vc_plummer(x), bounds=(0.1, 5), method='bounded',
                         options={'xatol': 1e-10}).x
print(f'numerical peak at r = {r_peak:.8f},  sqrt(2) b = {np.sqrt(2) * b:.8f}')
assert abs(r_peak - np.sqrt(2) * b) < 1e-6

**Figure 1.3.** Circular speed $v_c(r)$ of three models of unit mass: a point mass, a homogeneous sphere of radius 1 and a Plummer sphere with $b = 1$.

Real galaxies show *flat* rotation curves at large radius — neither of the shapes in Figure 1.3.
That observation is one of the main pieces of evidence for dark matter halos, which we
meet in Chapter 4.

## 1.3 Energy, angular momentum and the effective potential

In a spherical potential a star conserves its **energy** and its **angular-momentum vector**
(the force is radial, so it exerts no torque):

$$ E = \tfrac12 v^2 + \Phi(r), \qquad \mathbf L = \mathbf r \times \mathbf v. $$

Since $\mathbf L$ is fixed, the orbit stays in a plane. Split the velocity into a radial part
$v_r$ and a tangential part $v_t = L/r$:

$$ E = \tfrac12 v_r^2 + \underbrace{\Phi(r) + \frac{L^2}{2r^2}}_{\Phi_{\rm eff}(r)} . $$

This is a one-dimensional problem: a bead of energy $E$ sliding in the **effective potential**
$\Phi_{\rm eff}$. Because $v_r^2 \ge 0$, the star is confined to where $\Phi_{\rm eff}(r) \le E$.
The two radii where $\Phi_{\rm eff}(r) = E$ are the **pericentre** $r_p$ and **apocentre** $r_a$.
The minimum of $\Phi_{\rm eff}$ is the circular orbit with that $L$.

In [ ]:
def phi_kepler(r):  return -G * M / r
def phi_plummer(r): return -G * M / np.sqrt(r**2 + b**2)

def phi_eff(r, L, phi):
    """Effective potential Phi(r) + L^2 / (2 r^2). `phi` is a function, passed in like any variable."""
    return phi(r) + L**2 / (2 * r**2)

r = np.linspace(0.05, 5, 500)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for L in [0.4, 0.6, 0.8, 1.0]:
    ax[0].plot(r, phi_eff(r, L, phi_kepler), label=f'$L={L}$')
    ax[1].plot(r, phi_eff(r, L, phi_plummer), label=f'$L={L}$')
ax[0].plot(r, phi_kepler(r), 'k--', label=r'$\Phi$ ($L=0$)')
ax[1].plot(r, phi_plummer(r), 'k--', label=r'$\Phi$ ($L=0$)')
ax[0].set_ylim(-3, 1); ax[1].set_ylim(-1.1, 0.3)
ax[0].set_title('point mass'); ax[1].set_title('Plummer sphere')
for a in ax: a.set_xlabel('$r$'); a.set_ylabel(r'$\Phi_{\rm eff}$'); a.legend()
plt.show()

**Figure 1.4.** Effective potential $\Phi_{\rm eff}(r)$ for $L$ = 0.4–1.0 (coloured) and the potential itself ($L = 0$, black dashed), for a point mass (left) and the Plummer sphere (right).

The $L^2/2r^2$ term is a **centrifugal barrier**: the more angular momentum, the further the
star is kept from the centre. Note the difference at small $r$ in Figure 1.4: the Kepler potential plunges to
$-\infty$, so for small $L$ the well is deep and narrow; the Plummer potential has a finite core,
so the barrier always wins close to the centre.

### Turning points of a Kepler orbit

For a point mass the answer is known: an orbit with semi-major axis $a$ and eccentricity $e$
has $E = -GM/2a$, $L^2 = GMa(1-e^2)$, and turns at $r_{p,a} = a(1 \mp e)$.

We find the roots of $f(r) = \Phi_{\rm eff}(r) - E$ with `scipy.optimize.brentq(f, lo, hi)`. Brent's
method needs an interval $[{\rm lo}, {\rm hi}]$ on which $f$ changes sign, and then is guaranteed to
converge. For a bound orbit, $f > 0$ near $r = 0$ and far away, and $f < 0$ at the circular radius
$r_c$ (bottom of the well). So $[\epsilon, r_c]$ brackets the pericentre and $[r_c, r_{\max}]$ the
apocentre. For a point mass, $r_c = L^2/GM$.

*In AGAMA:* `pot.Rperiapo(E, L)` returns both turning points directly (Chapter 3).

In [ ]:
def turning_points(E, L, phi, r_circ, r_max=1e3):
    """Pericentre and apocentre: roots of Phi_eff(r) = E either side of the circular radius."""
    f = lambda r: phi_eff(r, L, phi) - E
    # xtol=1e-14: stop when the root is located to 1e-14 in r
    return brentq(f, 1e-8, r_circ, xtol=1e-14), brentq(f, r_circ, r_max, xtol=1e-14)

a, e = 1.0, 0.7
E = -G * M / (2 * a)
L = np.sqrt(G * M * a * (1 - e**2))
rp, ra = turning_points(E, L, phi_kepler, r_circ=L**2 / (G * M))
print(f'pericentre {rp:.12f}   expected a(1-e) = {a * (1 - e):.12f}')
print(f'apocentre  {ra:.12f}   expected a(1+e) = {a * (1 + e):.12f}')
assert abs(rp - a * (1 - e)) < 1e-8 * a and abs(ra - a * (1 + e)) < 1e-8 * a

r = np.linspace(0.1, 2.5, 400)
plt.figure(figsize=(6, 4))
plt.plot(r, phi_eff(r, L, phi_kepler), 'k-', label=r'$\Phi_{\rm eff}$')
plt.axhline(E, color='C1', label='$E$')
plt.plot([rp, ra], [E, E], 'o', color='C3', label='turning points')
plt.ylim(-2, 0.5); plt.xlabel('$r$'); plt.ylabel('energy'); plt.legend()
plt.title(f'Kepler orbit, $a={a}$, $e={e}$'); plt.show()

**Figure 1.5.** Effective potential of a Kepler orbit with $a = 1$, $e = 0.7$ (black), its energy $E$ (horizontal line) and the turning points $r_p$, $r_a$ (red points).

### The same in the Plummer potential

For the Plummer sphere there is no simple formula, but the method is identical. We pick a
star at $r=1$ moving tangentially (perpendicular to the radius) with 70% of the circular speed.
A tangential start means $v_r = 0$, so $r = 1$ is itself a turning point; with less than circular
speed the star falls inwards, so it is the apocentre.

The circular radius for this $L$ — needed to bracket the roots — is now found numerically as the
minimum of $\Phi_{\rm eff}$ (*in AGAMA:* `pot.Rcirc(L=...)`).

In [ ]:
r0, v0 = 1.0, 0.7 * vc_plummer(1.0)
L = r0 * v0                                   # |r x v| = r v for a tangential velocity
E = 0.5 * v0**2 + phi_plummer(r0)
r_circ = minimize_scalar(lambda r: phi_eff(r, L, phi_plummer), bounds=(1e-3, 10), method='bounded').x
rp, ra = turning_points(E, L, phi_plummer, r_circ)
print(f'circular radius for this L: {r_circ:.4f}')
print(f'pericentre {rp:.6f}, apocentre {ra:.6f}')
assert abs(ra - r0) < 1e-8          # we launched it tangentially, so r0 is a turning point

In Chapter 2 we will integrate this orbit numerically and check that it really oscillates
between these two radii.

## 1.4 The virial theorem

For a self-gravitating system in a steady state, the total kinetic energy $K$ and potential
energy $W$ obey

$$ 2K + W = 0 . $$

The potential energy of a spherical system is the work needed to assemble it shell by shell:
bringing a shell of mass $dm = 4\pi r^2 \rho\, dr$ in from infinity onto a sphere of mass $M(r)$
releases $G M(r)\, dm / r$, so

$$ W = -4\pi G \int_0^\infty \rho(r)\, M(r)\, r\, dr . $$

For the Plummer sphere $W = -\dfrac{3\pi}{32}\dfrac{GM^2}{b}$, so the virial theorem predicts
$K = \dfrac{3\pi}{64}\dfrac{GM^2}{b}$ and a mean-square speed $\langle v^2\rangle = 2K/M$.
This lets us estimate a system's mass from its size and velocity spread — the idea behind
the first evidence for dark matter in galaxy clusters (Zwicky 1933).

In [ ]:
# integrand rho(r) * M(r) * r, with the Plummer M(r) = M r^3 / (r^2 + b^2)^{3/2}
W_num = -4 * np.pi * G * quad(lambda x: rho_plummer(x) * M * x**3 / (x**2 + b**2)**1.5 * x, 0, np.inf)[0]
W_exact = -3 * np.pi / 32 * G * M**2 / b
print(f'W numerical {W_num:.10f},  formula {W_exact:.10f}')
assert abs(W_num / W_exact - 1) < 1e-6
print(f'virial prediction: K = {-W_exact / 2:.4f},  rms speed = {np.sqrt(-W_exact / M):.4f}')

We check the kinetic-energy side in Chapter 8, by sampling stellar velocities from the
Plummer model's distribution function.

## 1.5 The same with AGAMA

Everything above took a page of hand-written code. AGAMA does each step in one call, for any of
its built-in models or for a density profile you supply. Here we redo §1.2–1.3 with AGAMA and check
that it agrees with our own numbers — the same "trust after checking" rule, now applied to a library.

First install/import AGAMA (on Colab this takes about 10 seconds; see Chapter 0).

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

**Units.** We do *not* call `agama.setUnits` here. Without it AGAMA works in "N-body units" with
$G = 1$, exactly the units of this chapter, so its numbers can be compared directly with ours.

**Models.** `agama.Potential(type='Plummer', mass=M, scaleRadius=b)` is the analytic Plummer
sphere. A point mass (Kepler potential) has no separate type in AGAMA: it is a Plummer sphere with
`scaleRadius=0`, since $-GM/\sqrt{r^2+b^2} \to -GM/r$ as $b \to 0$.

**Inputs.** AGAMA evaluates quantities at Cartesian points, given as an array of shape `(N, 3)`.
For a spherical model any direction will do; we put the points on the $x$-axis.

In [ ]:
pot_plummer = agama.Potential(type='Plummer', mass=M, scaleRadius=b)
pot_kepler = agama.Potential(type='Plummer', mass=M, scaleRadius=0)     # point mass

r = np.logspace(-2, 2, 60)                       # the same radii as in §1.2
points = np.column_stack([r, 0 * r, 0 * r])      # shape (60, 3): points (r, 0, 0)

Phi_agama = pot_plummer.potential(points)        # Phi at each point
M_agama = pot_plummer.enclosedMass(r)            # mass inside each radius (takes radii, not points)
vc_agama = np.sqrt(-r * pot_plummer.force(points)[:, 0])   # v_c = sqrt(-R F_R); F_R = F_x on the x-axis

print(f'max |AGAMA / ours - 1|:  Phi {np.max(np.abs(Phi_agama / Phi_num - 1)):.1e},  '
      f'M(r) {np.max(np.abs(M_agama / M_num - 1)):.1e},  v_c {np.max(np.abs(vc_agama / vc_plummer(r) - 1)):.1e}')
assert np.allclose(Phi_agama, Phi_num, rtol=1e-6) and np.allclose(M_agama, M_num, rtol=1e-6)
assert np.allclose(vc_agama, vc_plummer(r), rtol=1e-10)

### Solving the Poisson equation for your own density

For the analytic Plummer model AGAMA just evaluates the formula. The real power is that it can
compute the potential of *any* density profile. `type='Multipole'` expands the density in spherical
harmonics and solves the Poisson equation $\nabla^2\Phi = 4\pi G\rho$ on a radial grid, then
interpolates with splines. We hand it our own `rho_plummer` and pretend we do not know the answer.

Two AGAMA conventions to note:
- a user density function receives an array of points of shape `(N, 3)` and must return `N`
  densities, so we wrap `rho_plummer`, which expects radii;
- `symmetry='spherical'` tells AGAMA the density depends only on $r$ (it cannot guess this from a
  Python function, and refuses to proceed without it).

In [ ]:
def rho_for_agama(xyz):
    return rho_plummer(np.linalg.norm(xyz, axis=1))   # radius of each of the N points, then rho(r)

pot_multipole = agama.Potential(type='Multipole', density=rho_for_agama, symmetry='spherical')
Phi_multipole = pot_multipole.potential(points)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].semilogx(r, Phi_exact, 'k-', lw=3, alpha=0.3, label='formula')
ax[0].semilogx(r, Phi_num, 'r--', label='ours (§1.2)')
ax[0].semilogx(r, Phi_multipole, 'b:', label='AGAMA Multipole')
ax[0].set_xlabel('$r$'); ax[0].set_ylabel(r'$\Phi(r)$'); ax[0].legend()
ax[1].loglog(r, np.abs(Phi_num / Phi_exact - 1), 'r-', label='ours (§1.2)')
ax[1].loglog(r, np.abs(Phi_multipole / Phi_exact - 1), 'b-', label='AGAMA Multipole')
ax[1].set_xlabel('$r$'); ax[1].set_ylabel('relative error in $\\Phi$'); ax[1].legend()
plt.tight_layout(); plt.show()

err_mp = np.max(np.abs(Phi_multipole / Phi_exact - 1))
print(f'max relative error of the Multipole potential: {err_mp:.1e}')
assert err_mp < 1e-5

**Figure 1.6.** Left: potential $\Phi(r)$ of the Plummer sphere from the formula (thick grey), our quadrature of §1.2 (red dashed) and AGAMA's Multipole expansion of the same density (blue dotted). Right: relative error of the two numerical potentials.

The Multipole potential is accurate to a few parts in a million. It is an *approximation*: AGAMA
solves the Poisson equation once on a grid of radii and then interpolates between grid points
with splines, and the error in Figure 1.6 (right) is that interpolation error. Our direct quadrature
(§1.2) is more accurate, but it needs two new integrals for every radius; the Multipole potential is
built once and then evaluated almost instantly anywhere — essential when integrating orbits, which
needs the force millions of times. It works equally well for densities with no analytic potential
at all, which is most realistic galaxy models.

### Circular radius and turning points

The effective-potential calculations of §1.3 are also built in:
- `pot.Rcirc(L=...)` — radius of the circular orbit with angular momentum $L$ (the minimum of $\Phi_{\rm eff}$);
- `pot.Rperiapo([E, L])` — pericentre and apocentre of an orbit with energy $E$ and angular
  momentum $L$ (the roots of $\Phi_{\rm eff} = E$). It also accepts an `(N, 2)` array to do many
  orbits at once.

In [ ]:
# Kepler orbit with a = 1, e = 0.7 (as in §1.3)
a, e = 1.0, 0.7
E_k, L_k = -G * M / (2 * a), np.sqrt(G * M * a * (1 - e**2))
rp_k, ra_k = pot_kepler.Rperiapo([E_k, L_k])
print(f'Kepler : AGAMA r_p, r_a = {rp_k:.10f}, {ra_k:.10f};  expected a(1-e), a(1+e) = {a*(1-e):.10f}, {a*(1+e):.10f}')

# Plummer orbit launched tangentially at r = 1 with 70% of v_c (as in §1.3); E, L, rp, ra, r_circ from there
rp_a, ra_a = pot_plummer.Rperiapo([E, L])
rc_a = pot_plummer.Rcirc(L=L)
print(f'Plummer: AGAMA r_p, r_a = {rp_a:.8f}, {ra_a:.8f};  ours = {rp:.8f}, {ra:.8f}')
print(f'Plummer: AGAMA R_circ   = {rc_a:.8f};              ours = {r_circ:.8f}')
assert abs(rp_k - a * (1 - e)) < 1e-8 and abs(ra_k - a * (1 + e)) < 1e-8
assert abs(rp_a - rp) < 1e-6 and abs(ra_a - ra) < 1e-6 and abs(rc_a - r_circ) < 1e-4

r_grid = np.linspace(0.3, 1.6, 300)
plt.figure(figsize=(6, 4))
plt.plot(r_grid, phi_eff(r_grid, L, phi_plummer), 'k-', label=r'$\Phi_{\rm eff}$ (ours)')
plt.axhline(E, color='C1', label='$E$')
plt.plot([rp, ra], [E, E], 'o', ms=10, mfc='none', color='C3', label='turning points (ours)')
plt.plot([rp_a, ra_a], [E, E], 'x', ms=8, color='C0', label='Rperiapo (AGAMA)')
plt.axvline(rc_a, color='gray', ls=':', label='Rcirc (AGAMA)')
plt.xlabel('$r$'); plt.ylabel('energy'); plt.title('Plummer orbit from §1.3'); plt.legend(); plt.show()

**Figure 1.7.** Effective potential of the Plummer orbit of §1.3 (black) and its energy $E$ (horizontal line). Circles: our turning points; crosses: AGAMA's `Rperiapo`; dotted line: AGAMA's `Rcirc`.

AGAMA agrees with our hand calculation (Figure 1.7) to the precision we asked for. (Our `r_circ` came from
`minimize_scalar` with its default tolerance, about $10^{-5}$, so that is the agreement there.)
From Chapter 3 onwards we use these AGAMA functions directly.

## Summary

- Only the mass inside $r$ pulls a star at $r$ inwards: $g = -GM(r)/r^2$.
- Density → enclosed mass → potential → circular speed, all by one-dimensional integrals.
- In a spherical potential, $E$ and $L$ reduce the orbit to motion in $\Phi_{\rm eff}(r)$;
  its turning points are the pericentre and apocentre.
- The virial theorem, $2K + W = 0$, links a system's mass, size and velocities.
- AGAMA reproduces all of these, for analytic models and for any density you supply.

| quantity | by hand (§1.1–1.4) | AGAMA (§1.5) |
|---|---|---|
| enclosed mass $M(r)$ | `quad` over $4\pi r^2\rho$ | `pot.enclosedMass(r)` |
| potential $\Phi$ | inner mass + outer shells | `pot.potential(points)`; any density: `type='Multipole'` |
| circular speed $v_c$ | $\sqrt{GM(r)/r}$ | `np.sqrt(-R * pot.force(points)[:, 0])` |
| circular radius for given $L$ | minimum of $\Phi_{\rm eff}$ | `pot.Rcirc(L=...)` |
| pericentre, apocentre | roots of $\Phi_{\rm eff} = E$ | `pot.Rperiapo(E, L)` |

## Exercises

1. Repeat §1.1 with a shell of 200 points. How large is the field inside now? Plot the error
   in $g$ against $N$ on log axes and find the scaling.
2. The **Hernquist** model has $\rho = \dfrac{M b}{2\pi r (r+b)^3}$. Compute $M(r)$ and $\Phi(r)$
   numerically with the functions above and compare with $M(r) = M r^2/(r+b)^2$,
   $\Phi = -GM/(r+b)$. Where does its circular speed peak?
3. For a homogeneous sphere of radius $a$, show that $\Phi = -\dfrac{GM}{2a^3}(3a^2 - r^2)$ inside.
   Plot $\Phi_{\rm eff}$ for $L = 0.3$ and find the turning points of an orbit with $E = -1.2$.
4. For the Plummer orbit in §1.3, find $L$ such that $r_a/r_p = 10$ for a star launched
   tangentially at $r = 1$.
5. Verify $W = -\tfrac{3\pi}{32} GM^2/b$ by integrating $\tfrac12\int \rho\,\Phi\, dV$ instead.
   Why does it give the same answer?
6. Build the Hernquist model of Exercise 2 in AGAMA in two ways — `type='Dehnen', gamma=1`
   (the Hernquist model is a Dehnen model with inner slope 1) and `type='Multipole'` from your own
   density function — and compare their potentials with the formula. Then use `Rperiapo` to
   find the turning points of an orbit with $E = -0.4$, $L = 0.3$.

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §2.2 and §3.1.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapters 2–4: https://galaxiesbook.org
- H. C. Plummer (1911), MNRAS 71, 460: https://doi.org/10.1093/mnras/71.5.460
- F. Zwicky (1933), Helvetica Physica Acta 6, 110: https://ui.adsabs.harvard.edu/abs/1933AcHPh...6..110Z